# SVHN Benchmark: MLP, RNN, CNN on Colab & Kaggle


## How to use
1. **Select accelerator** in your notebook settings (GPU for GPU runs). The notebook can still force **CPU** even on a GPU runtime.
2. Edit the **parameters** in the next cell. For the full factorial at one platform, use `fractions = [0.5, 1.0]` and keep both `cpu` and `gpu` in `hardware_list`.
3. Results append to `kaggle_svhn_benchmark_results.csv`.


In [1]:
# Environment & platform detection
from datetime import datetime, timezone
import os, sys, platform, random, time, subprocess
import numpy as np
import torch

def detect_platform():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ: return 'KAGGLE'
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ: return 'COLAB'
    return 'LOCAL'

def get_cpu_name():
    try:
        # Retrieves the specific CPU model name on Linux (Colab/Kaggle)
        return subprocess.check_output("grep -m 1 'model name' /proc/cpuinfo | cut -d: -f2", shell=True).decode().strip()
    except Exception:
        # Fallback for other environments
        return platform.processor() or "Unknown CPU"

PLATFORM = detect_platform()
CPU_NAME = get_cpu_name()
CUDA_AVAILABLE = torch.cuda.is_available()
AUTO_DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

print('Platform:', PLATFORM)
print('CPU Model:', CPU_NAME)
print('CUDA available:', CUDA_AVAILABLE, '| Auto device:', AUTO_DEVICE)
if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))

Platform: KAGGLE
CPU Model: Intel(R) Xeon(R) CPU @ 2.00GHz
CUDA available: True | Auto device: cuda
GPU: Tesla T4


In [2]:
# Ensure SciPy (required by torchvision.datasets.SVHN) is available
try:
    import scipy
except Exception:
    print('Installing SciPy...')
    %pip -q install scipy
    import scipy


In [3]:
# ==== Experiment parameters (edit here) ====
fractions = [0.5, 1.0]   # dataset sizes to benchmark (50% vs 100%)
epochs = 5
batch_size = 64
repeats = 1
num_workers = 2
plots = True
save_metrics = 'kaggle_svhn_benchmark_results.csv'
seed = 42
models_to_run = ['mlp', 'rnn', 'cnn']
hardware_list = ['cpu', 'gpu']
out_dir = 'outputs'
data_dir = './data'


In [4]:
# Imports & helpers
import os, csv, math
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def set_seeds(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s);
    torch.cuda.manual_seed_all(s);
    torch.backends.cudnn.deterministic = True;
    torch.backends.cudnn.benchmark = False

class MLPClassifier(nn.Module):
    def __init__(self, input_size=3*32*32, hidden_units=256, num_classes=10):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, num_classes)
        )
    def forward(self, x): return self.model(x)

class RNNClassifier(nn.Module):
    def __init__(self, input_size=96, hidden_units=128, num_layers=1, num_classes=10):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_units, num_layers=num_layers, batch_first=True, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_units, num_classes)
    def forward(self, x):
        x = x.permute(0,2,3,1)              # (B,C,H,W)->(B,H,W,C)
        x = x.reshape(x.size(0), 32, 32*3)  # seq_len=32, features=96
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.fc(last)

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.drop = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64*8*8, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = torch.flatten(x,1); x = self.drop(x)
        x = F.relu(self.fc1(x)); return self.fc2(x)

def get_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomGrayscale(p=0.1),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
    ])

def make_datasets(root, dataset_fraction, seed):
    tfm = get_transforms()
    full_train = datasets.SVHN(root=root, split='train', transform=tfm, download=True)
    test_ds = datasets.SVHN(root=root, split='test', transform=tfm, download=True)
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(full_train), generator=g).tolist()
    take = int(len(idx) * dataset_fraction)
    sub = Subset(full_train, idx[:take])
    trn_size = int(0.8 * len(sub))
    val_size = len(sub) - trn_size
    train_ds, val_ds = random_split(sub, [trn_size, val_size], generator=g)
    return train_ds, val_ds, test_ds

def get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers):
    pin = (device_kind=='gpu')
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    return train_loader, val_loader, test_loader

def train_one_epoch(model, device, loader, optimizer, criterion):
    model.train(); run_loss=0.0; correct=0; total=0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        run_loss += loss.item()*x.size(0)
        _,pred = torch.max(out,1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def evaluate(model, device, loader, criterion):
    model.eval(); run_loss=0.0; correct=0; total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            run_loss += loss.item()*x.size(0)
            _,pred = torch.max(out,1)
            correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def plot_losses(train_losses, val_losses, out_path):
    plt.figure(figsize=(10,4)); plt.plot(train_losses,label='Train'); plt.plot(val_losses,label='Val');
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout(); plt.savefig(out_path); plt.close()

def plot_confusion(model, device, loader, classes, out_path):
    model.eval(); P=[]; T=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); out=model(x); P.extend(out.argmax(1).cpu().numpy().tolist()); T.extend(y.numpy().tolist())
    cm = confusion_matrix(T,P); disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(6,6)); disp.plot(xticks_rotation='vertical', ax=ax, cmap='Blues', colorbar=False)
    plt.tight_layout(); plt.savefig(out_path); plt.close()


In [5]:
# Benchmark loop: runs (hardware × fractions × models × repeats) and logs CSV
from pathlib import Path
Path(out_dir).mkdir(parents=True, exist_ok=True)
classes = [str(i) for i in range(10)]

header = ['timestamp','platform_hint','device_kind','device_name','torch_version','cuda_available','cuda_version',
          'model','dataset_fraction','epochs','batch_size','train_samples','val_samples','test_samples',
          'repeat_idx','seed','total_train_time_s','per_epoch_time_s_mean','throughput_samples_per_s',
          'final_train_loss','final_train_acc','final_val_loss','final_val_acc','test_loss','test_acc']
write_header = not os.path.exists(save_metrics)
with open(save_metrics, 'a', newline='') as f:
    wr = csv.writer(f)
    if write_header: wr.writerow(header)

    for hw in hardware_list:
        device_kind = 'gpu' if hw=='gpu' and torch.cuda.is_available() else ('cpu' if hw=='cpu' else 'skip')
        if hw=='gpu' and not torch.cuda.is_available():
            print('[SKIP] GPU selected but CUDA not available.')
            continue
        device = torch.device('cuda') if device_kind=='gpu' else torch.device('cpu')
        name = torch.cuda.get_device_name(0) if device_kind=='gpu' else CPU_NAME
        print('=== Hardware: {} ({}) ==='.format(device_kind.upper(), name))

        for frac in fractions:
            set_seeds(seed)
            train_ds, val_ds, test_ds = make_datasets(data_dir, frac, seed)
            train_loader, val_loader, test_loader = get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers)

            for model_name in models_to_run:
                for rep in range(1, repeats+1):
                    set_seeds(seed + rep - 1)
                    if model_name=='mlp': model = MLPClassifier().to(device)
                    elif model_name=='rnn': model = RNNClassifier().to(device)
                    elif model_name=='cnn': model = CNNClassifier().to(device)
                    else: raise ValueError(model_name)

                    criterion = nn.CrossEntropyLoss()
                    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
                    epoch_times = []
                    tr_losses, va_losses = [], []

                    t0 = time.perf_counter()
                    for ep in range(1, epochs+1):
                        e0 = time.perf_counter()
                        tr_loss, tr_acc = train_one_epoch(model, device, train_loader, optim, criterion)
                        if device_kind=='gpu': torch.cuda.synchronize()
                        e1 = time.perf_counter(); epoch_times.append(e1-e0)
                        va_loss, va_acc = evaluate(model, device, val_loader, criterion)
                        tr_losses.append(tr_loss); va_losses.append(va_loss)
                        print('[{}][Rep {}] Epoch {}/{} | Train {:.4f}/{:.2f}% | Val {:.4f}/{:.2f}% | Time {:.2f}s'.format(
                              model_name.upper(), rep, ep, epochs, tr_loss, tr_acc, va_loss, va_acc, epoch_times[-1]))

                    if device_kind=='gpu': torch.cuda.synchronize()
                    total_train_time = time.perf_counter() - t0
                    test_loss, test_acc = evaluate(model, device, test_loader, criterion)
                    per_epoch_mean = float(np.mean(epoch_times)) if epoch_times else 0.0
                    throughput = (len(train_loader.dataset)*epochs)/total_train_time if total_train_time>0 else 0.0

                    # Optional plots
                    if plots:
                        base = os.path.join(out_dir, '{}_{}_rep{}_frac{}'.format(model_name, device_kind, rep, frac))
                        plot_losses(tr_losses, va_losses, base + '_loss.png')
                        plot_confusion(model, device, val_loader, classes, base + '_confusion_val.png')

                    wr.writerow([
                        datetime.now(timezone.utc).isoformat(),
                        os.environ.get('KAGGLE_KERNEL_RUN_TYPE','COLAB' if 'COLAB_GPU' in os.environ else 'LOCAL'),
                        device_kind, name, torch.__version__, torch.cuda.is_available(),
                        (torch.version.cuda if torch.version.cuda is not None else ''),
                        model_name, frac, epochs, batch_size, len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset),
                        rep, seed + rep - 1, round(total_train_time,4), round(per_epoch_mean,4), round(throughput,2),
                        round(tr_losses[-1],4), round(tr_acc,2), round(va_losses[-1],4), round(va_acc,2), round(test_loss,4), round(test_acc,2)
                    ])
                    f.flush()
                    print('[DONE] {} | HW {} | Rep {} | Total {:.2f}s | Throughput {:.2f} samples/s | ValAcc {:.2f}% | TestAcc {:.2f}%'.format(
                          model_name.upper(), device_kind, rep, total_train_time, throughput, va_acc, test_acc))

print('Results appended to:', save_metrics)


=== Hardware: CPU (Intel(R) Xeon(R) CPU @ 2.00GHz) ===


100%|██████████| 182M/182M [00:04<00:00, 36.9MB/s] 
100%|██████████| 64.3M/64.3M [00:04<00:00, 15.4MB/s]


[MLP][Rep 1] Epoch 1/5 | Train 1.4478/51.80% | Val 1.0376/67.79% | Time 7.25s
[MLP][Rep 1] Epoch 2/5 | Train 0.9771/69.30% | Val 0.9162/71.10% | Time 7.53s
[MLP][Rep 1] Epoch 3/5 | Train 0.8418/73.87% | Val 0.8706/72.97% | Time 7.06s
[MLP][Rep 1] Epoch 4/5 | Train 0.7548/76.28% | Val 0.7879/76.14% | Time 7.10s
[MLP][Rep 1] Epoch 5/5 | Train 0.6824/78.62% | Val 0.7629/76.58% | Time 7.14s
[DONE] MLP | HW cpu | Rep 1 | Total 43.63s | Throughput 3358.15 samples/s | ValAcc 76.58% | TestAcc 73.08%
[RNN][Rep 1] Epoch 1/5 | Train 2.2045/20.57% | Val 2.1327/23.82% | Time 8.05s
[RNN][Rep 1] Epoch 2/5 | Train 1.9785/28.99% | Val 1.8398/35.59% | Time 7.86s
[RNN][Rep 1] Epoch 3/5 | Train 1.7423/38.60% | Val 1.6001/43.90% | Time 8.28s
[RNN][Rep 1] Epoch 4/5 | Train 1.5447/46.57% | Val 1.4913/47.64% | Time 7.94s
[RNN][Rep 1] Epoch 5/5 | Train 1.4228/51.18% | Val 1.3285/54.87% | Time 8.10s
[DONE] RNN | HW cpu | Rep 1 | Total 48.83s | Throughput 3000.11 samples/s | ValAcc 54.87% | TestAcc 52.33%
[CNN][

In [6]:
# Summarize results: mean±std for time and accuracy, and GPU speedup
import pandas as pd
df = pd.read_csv(save_metrics)
display(df.tail(10))

# Grouped summary
grp = df.groupby(['platform_hint','device_kind','model','dataset_fraction']).agg(
    runs=('repeat_idx','count'),
    total_time_mean=('total_train_time_s','mean'), total_time_std=('total_train_time_s','std'),
    per_epoch_mean=('per_epoch_time_s_mean','mean'),
    throughput_mean=('throughput_samples_per_s','mean'),
    val_acc_mean=('final_val_acc','mean'), val_acc_std=('final_val_acc','std'),
    test_acc_mean=('test_acc','mean'), test_acc_std=('test_acc','std')
).reset_index()
display(grp.sort_values(['model','dataset_fraction','device_kind']))

# Compute GPU speedup vs CPU by matching rows
pivot = grp.pivot_table(index=['platform_hint','model','dataset_fraction'], columns='device_kind', values='total_time_mean')
if 'cpu' in pivot.columns and 'gpu' in pivot.columns:
    pivot['gpu_speedup'] = pivot['cpu'] / pivot['gpu']
    display(pivot)
else:
    print('GPU or CPU results missing; run both hardware configs.')


,timestamp,platform_hint,device_kind,device_name,torch_version,cuda_available,cuda_version,model,dataset_fraction,epochs,...,seed,total_train_time_s,per_epoch_time_s_mean,throughput_samples_per_s,final_train_loss,final_train_acc,final_val_loss,final_val_acc,test_loss,test_acc
2,2026-03-09T00:50:01.515429+00:00,Interactive,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.9.0+cu126,True,12.6,cnn,0.5,5,...,42,134.5266,23.8637,1089.08,0.3940,88.24,0.4145,88.25,0.4875,86.33
3,2026-03-09T00:51:39.063840+00:00,Interactive,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.9.0+cu126,True,12.6,mlp,1.0,5,...,42,85.0369,14.0470,3445.86,0.5915,81.97,0.6973,79.60,0.8570,76.51
4,2026-03-09T00:53:25.396739+00:00,Interactive,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.9.0+cu126,True,12.6,rnn,1.0,5,...,42,96.8258,16.1647,3026.31,1.2060,58.72,1.2412,57.31,1.3081,55.80
5,2026-03-09T00:58:15.076591+00:00,Interactive,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.9.0+cu126,True,12.6,cnn,1.0,5,...,42,272.9752,48.6167,1073.45,0.3520,89.52,0.3583,89.52,0.4057,88.07
6,2026-03-09T00:59:01.235334+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,mlp,0.5,5,...,42,35.9922,5.7934,4070.60,0.6849,78.61,0.7561,76.95,0.9724,72.96
7,2026-03-09T00:59:43.349942+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,rnn,0.5,5,...,42,35.4561,5.6829,4132.15,1.5219,47.27,1.4849,48.65,1.5368,47.73
8,2026-03-09T01:00:28.060007+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,cnn,0.5,5,...,42,37.7309,6.0705,3883.02,0.3977,88.21,0.4351,87.50,0.5028,85.08
9,2026-03-09T01:01:51.476959+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,mlp,1.0,5,...,42,72.1388,11.5731,4061.96,0.5948,81.72,0.6729,80.33,0.8352,77.04
10,2026-03-09T01:03:10.959604+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,rnn,1.0,5,...,42,71.4084,11.4479,4103.51,1.1967,60.29,1.3479,55.66,1.4532,52.09
11,2026-03-09T01:04:32.504512+00:00,Interactive,gpu,Tesla T4,2.9.0+cu126,True,12.6,cnn,1.0,5,...,42,73.0788,11.7769,4009.71,0.3475,89.65,0.3505,89.92,0.4132,87.89


,platform_hint,device_kind,model,dataset_fraction,runs,total_time_mean,total_time_std,per_epoch_mean,throughput_mean,val_acc_mean,val_acc_std,test_acc_mean,test_acc_std
0,Interactive,cpu,cnn,0.5,1,134.5266,NaN,23.8637,1089.08,88.25,NaN,86.33,NaN
6,Interactive,gpu,cnn,0.5,1,37.7309,NaN,6.0705,3883.02,87.50,NaN,85.08,NaN
1,Interactive,cpu,cnn,1.0,1,272.9752,NaN,48.6167,1073.45,89.52,NaN,88.07,NaN
7,Interactive,gpu,cnn,1.0,1,73.0788,NaN,11.7769,4009.71,89.92,NaN,87.89,NaN
2,Interactive,cpu,mlp,0.5,1,43.6282,NaN,7.2153,3358.15,76.58,NaN,73.08,NaN
8,Interactive,gpu,mlp,0.5,1,35.9922,NaN,5.7934,4070.60,76.95,NaN,72.96,NaN
3,Interactive,cpu,mlp,1.0,1,85.0369,NaN,14.0470,3445.86,79.60,NaN,76.51,NaN
9,Interactive,gpu,mlp,1.0,1,72.1388,NaN,11.5731,4061.96,80.33,NaN,77.04,NaN
4,Interactive,cpu,rnn,0.5,1,48.8348,NaN,8.0441,3000.11,54.87,NaN,52.33,NaN
10,Interactive,gpu,rnn,0.5,1,35.4561,NaN,5.6829,4132.15,48.65,NaN,47.73,NaN


device_kind                                cpu      gpu  gpu_speedup
platform_hint model dataset_fraction                                
Interactive   cnn   0.5               134.5266  37.7309     3.565423
                    1.0               272.9752  73.0788     3.735354
              mlp   0.5                43.6282  35.9922     1.212157
                    1.0                85.0369  72.1388     1.178796
              rnn   0.5                48.8348  35.4561     1.377331
                    1.0                96.8258  71.4084     1.355944

---
### Notes
- TorchVision's SVHN loader remaps the digit '0' to label **0** (the original dataset uses **10** for '0'), keeping class labels in **[0..9]** for `CrossEntropyLoss`.
- Loading SVHN `.mat` files via TorchVision requires **SciPy**.
- You can benchmark both **CPU and GPU** in a single GPU-enabled runtime; CPU runs are forced by selecting `device='cpu'`.